# Edith Demo Runtime

This notebook runs Edith continuously in **demo simulation mode** and publishes heartbeat, signal, and trade telemetry for the Streamlit dashboard.

It does not connect to MT5 or submit broker orders. Stop the loop with **Kernel → Interrupt**.


In [ ]:
from __future__ import annotations

import json
import os
import random
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

REPOSITORY_ROOT = Path.cwd()
DATA_DIR = REPOSITORY_ROOT / "data"
STATUS_PATH = DATA_DIR / "runtime_status.json"
SIGNALS_PATH = DATA_DIR / "signals.jsonl"
TRADES_PATH = DATA_DIR / "demo_trades.jsonl"

EXECUTION_MODE = os.getenv("LILITH_EXECUTION_MODE", "simulation").strip().lower()
if EXECUTION_MODE not in {"simulation", "demo"}:
    raise RuntimeError("Edith notebook loop is restricted to simulation/demo mode. Set LILITH_EXECUTION_MODE=simulation.")

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository: {REPOSITORY_ROOT}")
print(f"Execution mode: {EXECUTION_MODE}")
print("Telemetry paths initialised.")


In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def atomic_write_json(path: Path, payload: dict) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, sort_keys=True, indent=2), encoding="utf-8")
    temporary.replace(path)

def append_jsonl(path: Path, payload: dict) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, sort_keys=True, separators=(",", ":")))
        handle.write("\n")

def publish_status(**updates) -> dict:
    current = {}
    if STATUS_PATH.exists():
        try:
            current = json.loads(STATUS_PATH.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            current = {}
    current.update(updates)
    current["heartbeat_at"] = utc_now()
    atomic_write_json(STATUS_PATH, current)
    return current


In [ ]:
POLL_SECONDS = max(1, int(os.getenv("EDITH_DEMO_POLL_SECONDS", "5")))
MAX_ITERATIONS = int(os.getenv("EDITH_DEMO_MAX_ITERATIONS", "0"))
SYMBOL = os.getenv("EDITH_DEMO_SYMBOL", "XAUUSDm")
TIMEFRAME = os.getenv("EDITH_DEMO_TIMEFRAME", "M5")

session_id = str(uuid.uuid4())
iteration = 0
signals_seen = 0
demo_trades = 0
last_signal = "HOLD"
rng = random.Random(260729)

publish_status(connection="Online", runtime="running", mode="demo-simulation", session_id=session_id, symbol=SYMBOL, timeframe=TIMEFRAME, started_at=utc_now(), iteration=0, signals_seen=0, demo_trades=0, last_signal=last_signal, message="Edith demo loop started.")
print(f"Edith demo loop started: {session_id}")
print(f"Polling every {POLL_SECONDS}s. Interrupt the kernel to stop.")

try:
    while MAX_ITERATIONS <= 0 or iteration < MAX_ITERATIONS:
        iteration += 1
        score = round(rng.uniform(35, 92), 2)
        if score >= 72:
            side = "BUY" if iteration % 2 else "SELL"
            decision = "ENTER_DEMO"
        elif score >= 55:
            side = "WATCH"
            decision = "MONITOR"
        else:
            side = "HOLD"
            decision = "SKIP"

        signal = {"timestamp": utc_now(), "session_id": session_id, "iteration": iteration, "symbol": SYMBOL, "timeframe": TIMEFRAME, "signal": side, "decision": decision, "score": score, "mode": "demo-simulation", "reason": "deterministic demo signal for dashboard telemetry"}
        append_jsonl(SIGNALS_PATH, signal)
        signals_seen += 1
        last_signal = side

        if decision == "ENTER_DEMO":
            demo_trades += 1
            pnl = round(rng.uniform(-1.25, 2.50), 2)
            trade = {"trade_id": f"DEMO-{session_id[:8]}-{demo_trades:04d}", "timestamp": utc_now(), "session_id": session_id, "iteration": iteration, "symbol": SYMBOL, "timeframe": TIMEFRAME, "side": side, "status": "closed_demo", "entry_type": "simulated", "net_realised_pnl": pnl, "r_multiple": round(pnl / 1.25, 2), "mode": "demo-simulation"}
            append_jsonl(TRADES_PATH, trade)

        status = publish_status(connection="Online", runtime="running", mode="demo-simulation", session_id=session_id, symbol=SYMBOL, timeframe=TIMEFRAME, iteration=iteration, signals_seen=signals_seen, demo_trades=demo_trades, last_signal=last_signal, last_score=score, message="Telemetry received.")
        print(f"[{status['heartbeat_at']}] iteration={iteration} signal={last_signal} score={score} demo_trades={demo_trades}")
        time.sleep(POLL_SECONDS)

except KeyboardInterrupt:
    publish_status(connection="Offline", runtime="stopped", mode="demo-simulation", session_id=session_id, iteration=iteration, signals_seen=signals_seen, demo_trades=demo_trades, last_signal=last_signal, stopped_at=utc_now(), message="Notebook loop stopped by operator.")
    print("Edith demo loop stopped cleanly.")
except Exception as exc:
    publish_status(connection="Error", runtime="failed", mode="demo-simulation", session_id=session_id, iteration=iteration, signals_seen=signals_seen, demo_trades=demo_trades, last_signal=last_signal, error=repr(exc), message="Notebook loop failed.")
    raise
